# RIIMA: imagens, afinidade e interpretação

Um percurso em 13 etapas com as funções de `riima.core`, também usadas pela GUI e pelo CLI.
Cada imagem contribui igualmente com $[L^*,a^*,b^*,T]$: $D=4N$.
A referência serve somente para registro geométrico. **Cluster não é mineral**;
nomes, uniões e famílias são escolhas manuais do usuário.

Instale `requirements/notebook.txt` e execute as células em ordem, a partir da raiz
do projeto ou de `notebooks/`. O exemplo gera duas imagens geométricas sintéticas na pasta temporária de saída.
Os testes fornecem imagens sintéticas pequenas pelos parâmetros `RIIMA_*` da primeira célula.
O exemplo distribuído usa imagens geométricas sintéticas, sem identidade mineralógica. Elas permitem executar as etapas sem dados privados de uma lâmina.


## 1. Carregar imagens

Edite `paths` para seu conjunto de imagens. PPL e XPL são apenas exemplos de aquisição;
nenhuma modalidade recebe tratamento especial. A saída fica em uma pasta temporária,
cujo caminho é exibido abaixo, ou no diretório definido em `RIIMA_OUTPUT_DIR`.

In [ ]:
import sys
import tempfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / "riima" / "__init__.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from riima import core
from riima.report_plot_style import matplotlib_style

output = Path(globals().get("RIIMA_OUTPUT_DIR") or tempfile.mkdtemp(prefix="riima_notebook_"))
if "RIIMA_INPUT_PATHS" in globals():
    paths = [str(p) for p in RIIMA_INPUT_PATHS]
else:
    import importlib.util
    spec = importlib.util.spec_from_file_location("riima_example", ROOT / "scripts/generate_example.py")
    example = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(example)
    inputs = output / "inputs"
    example.write_fields(inputs)
    paths = [str(inputs / name) for name in ("field_a.png", "field_b.png")]
cfg = globals().get("RIIMA_CONFIG", core.Config(clusters=8, sample_pixels=80_000, seed=13))
reference_index = 0
mask_mode = globals().get("RIIMA_MASK_MODE", "auto")
originals = core.load_images(paths)

def show_panels(arrays, titles, cmap=None):
    with plt.rc_context(matplotlib_style()):
        fig, axes = plt.subplots(1, len(arrays), figsize=(4 * len(arrays), 3), squeeze=False)
        for ax, array, title in zip(axes.flat, arrays, titles):
            ax.imshow(array, cmap=cmap)
            ax.set_title(title)
            ax.axis("off")
        fig.tight_layout()
        plt.show()

show_panels(originals, [Path(p).name for p in paths])
print("RIIMA", core.__version__, "| saída:", output)

## 2. Registro

Cada imagem é registrada diretamente na referência. Inspecione o alinhamento e a
área comum antes de interpretar os resultados.

In [ ]:
registered = core.register_images(paths, reference_index=reference_index)
images = [item["rgb"] for item in registered["images"]]
overlap = registered["common_overlap"]
overlap_status = core.check_common_overlap(overlap, cfg)
show_panels(images, [f"Registrada: {Path(p).name}" for p in paths])
for item in registered["images"]:
    info = item["registration"]
    print(item["name"], info["method"], "correlação:", round(info["score_after"], 3))

## 3. Máscara

A máscara combina sobreposição geométrica, suporte da amostra e exclusão de anotações.
As duas últimas regras são heurísticas: revise a máscara. `overlap_only` é uma escolha
explícita para usar somente a sobreposição quando essas heurísticas não forem adequadas.

In [ ]:
valid = core.build_analysis_mask(images, overlap, mask_mode=mask_mode)
assert valid.any(), "Nenhum pixel válido; revise registro e máscara."
show_panels([valid], [f"Área válida: {100 * valid.mean():.1f}%"], cmap="gray")

## 4. CIELAB

Os canais usam unidades canônicas: $L^*\in[0,100]$, com $a^*$ e $b^*$ centrados em zero.
A mesma suavização é aplicada a todas as imagens.

In [ ]:
lab = [core.lab_star_channels(image, cfg.feature_blur_sigma) for image in images]
show_panels(list(lab[0]), ["L*", "a*", "b*"], cmap="viridis")

## 5. Heterogeneidade cromática local $T$

$T_i(p)=\sqrt{\sigma^2_{L_i}(p)+\sigma^2_{a_i}(p)+\sigma^2_{b_i}(p)}$.
Esta medida descreve variação local da cor, com a mesma janela fixa para cada imagem.

In [ ]:
heterogeneity = [core.local_chromatic_heterogeneity(*channels) for channels in lab]
show_panels(heterogeneity, [f"T: {Path(p).name}" for p in paths], cmap="magma")

## 6. Vetor multi-imagem $4N$

$x(p)=[L_1^*,a_1^*,b_1^*,T_1,\ldots,L_N^*,a_N^*,b_N^*,T_N]$.
Empilhamos os canais já calculados na ordem definida pelo núcleo, sem privilegiar a referência.

In [ ]:
sources = [plane for channels, t in zip(lab, heterogeneity) for plane in (*channels, t)]
assert len(sources) == 4 * len(images)
print(f"N = {len(images)}; D = {len(sources)}")

## 7. Normalização robusta

$z_j=(x_j-\operatorname{mediana}(x_j))/\operatorname{IQR}(x_j)$, estimados na amostra
de pixels válidos. IQR quase nulo recebe escala 1. Os mesmos parâmetros são usados
na predição de todos os pixels.

In [ ]:
sample_X, sample_idx, center, scale = core.standardized_sample(
    sources, valid, cfg.sample_pixels, cfg.seed,
)
print("Amostra normalizada:", sample_X.shape)
print("Medianas:", np.round(center, 2))
print("IQR (com proteção):", np.round(scale, 2))

## 8. Afinidade

Afinidade é proximidade no espaço normalizado: distância euclidiana e pesos unitários.
Não é necessário construir uma matriz de distâncias entre todos os pixels.
O exemplo abaixo mede a distância entre dois vetores da amostra.

In [ ]:
distance = np.linalg.norm(sample_X[0] - sample_X[1])
print(f"Distância euclidiana entre dois pixels amostrados: {distance:.3f}")

## 9. K-means

K-means ajusta os centros na amostra normalizada. O diagnóstico opcional reutiliza
essa amostra e calcula inércia, silhueta, Calinski-Harabasz e Davies-Bouldin.
O maior valor de silhueta sugere K; a sugestão não altera `cfg.clusters` nem executa nova análise.

In [ ]:
sample_labels, centers = core.fit_kmeans(sample_X, cfg.clusters, cfg.kmeans_attempts, cfg.seed)
RUN_K_DIAGNOSTIC = False  # Ative somente quando quiser comparar valores de K.
if RUN_K_DIAGNOSTIC:
    scores = core.evaluate_cluster_counts(sample_X, [3, 4, 6, 8], seed=cfg.seed)
    suggested = core.recommend_cluster_count(scores["k_values"], scores["silhouette"])
    print(scores)
    print(f"K = {suggested} (sugerido)")
print("K usado:", cfg.clusters)

## 10. Cluster map

O núcleo prediz os pixels e aplica a mesma regularização de pixels incertos usada
na GUI. Esse resultado é a solução bruta preservada, anterior à interpretação manual.
A confiança mede separação relativa entre os dois centros mais próximos, não certeza mineralógica.

In [ ]:
flat_labels, flat_confidence = core.predict_chunked(sources, center, scale, centers, cfg.chunk_pixels)
confidence = flat_confidence.reshape(valid.shape)
edges = core.edge_strength_multi(images)
raw_labels = core.confidence_regularize(
    flat_labels.reshape(valid.shape), confidence, edges, cfg.clusters, cfg, valid,
)
raw_labels = core.cleanup_uncertain_islands(raw_labels, confidence, edges, valid, cfg.clusters, cfg)
raw_labels.setflags(write=False)
palette = core.cluster_palette(cfg.clusters)
# Estatísticas descritivas nas unidades originais dos atributos.
feature_means = np.asarray([
    [float(plane[valid & (raw_labels == k)].mean()) if np.any(valid & (raw_labels == k)) else 0.0
     for plane in sources] for k in range(cfg.clusters)
])
optical = dict(labels=raw_labels, valid=valid, confidence=confidence, clusters=cfg.clusters,
               palette=palette, centers=centers, feature_means=feature_means,
               sample_X=sample_X, center=center, scale=scale, images=images,
               image_names=[item["name"] for item in registered["images"]],
               registered_images=registered["images"], reference_index=reference_index,
               reference=images[reference_index], overlap=overlap, overlap_status=overlap_status,
               mask_mode=mask_mode, output=str(output), feature_blur_sigma=cfg.feature_blur_sigma)
show_panels([core.render_optical_clusters(raw_labels, valid, palette)], ["Clusters brutos"])

## 11. Interpretação de clusters

Revise os clusters e consolide aqueles que representam a mesma interpretação petrográfica.
**Exemplo didático manual:** os IDs 0 e 1 abaixo são escolhidos apenas para demonstrar
a edição, sem justificativa mineralógica. Substitua a seleção após examinar suas imagens.
Os IDs são índices a partir de zero. `undo()` desfaz; `reset()` restaura toda a interpretação
sem recalcular clustering.

In [ ]:
state = core.Interpretation(optical)
state.merge_clusters([0, 1])
state.rename_cluster(0, "Associação de exemplo")
state.recolor_cluster(0, (100, 160, 190))
interpreted = state.interpreted()
assert np.array_equal(state.raw["labels"], raw_labels)
show_panels([core.render_optical_clusters(interpreted["labels"], valid, interpreted["palette"])],
            ["Clusters interpretados: exemplo manual"])

## 12. Famílias

Uma família associa um ou mais clusters por decisão do usuário. A seleção usa os
índices dos clusters interpretados atuais; a família conserva os IDs brutos correspondentes.
Este exemplo agrupa os clusters interpretados 0 e 1 sob um nome neutro.
Áreas sem associação permanecem com ID 0 no mapa de famílias.

In [ ]:
state.create_family([0, 1], name="Família de exemplo", color=(220, 170, 70))
families = state.family_result()
family_colors = np.vstack([np.array([[30, 30, 30]], dtype=np.uint8), families["palette"]])
show_panels([family_colors[families["family_map"]]], ["Família de exemplo: associação manual"])
print(families["stats"])

## 13. Exportação

A exportação usa os arrays e a interpretação já calculados. Não registra imagens nem
ajusta K-means novamente. São preservados os mapas brutos, interpretados e de famílias,
a máscara, a confiança, os parâmetros, a interpretação e as estatísticas.
Os nomes de exemplo não constituem identificação mineralógica.

In [ ]:
result = dict(output=str(output), optical=optical, config=cfg, interpretation=state)
from riima.exports import export_batch
with export_batch():
    core.save_session_outputs(result)
    core.save_registration_csv(registered["images"], output)
print("Arquivos exportados:")
print("\n".join(path.name for path in sorted(output.iterdir()) if path.is_file()))